<a href="https://colab.research.google.com/github/DanDC25/Statistical-Learning-with-Python/blob/main/%5BPractice_Logistic_%2B_Naive_Bayes_%2B_LDA_%2B_CV_score_%2B_1SE%5D_DSA211_Simulated_Q2_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to use this question

This question is built from **Lab 5 (Logistic Regression)** on the same `Carseats` data, together with the cross-validation workflow from **Lab 7, Part 4** and the one-test-touch step from **Lab 9**. Every step has been done in a lab.

- Work like the exam: about **18 minutes**, open book, no AI.
- For each part, find the matching lab cell, adapt the code, run it, then write your answer in the markdown cell under the part label.
- Keep to the sentence limits. A printed number with no sentence earns at most half the marks.
- Importing the libraries you need is part of the task. The **📍 Where to look** note under each part says which lab cell to open, including the Setup cells with the imports. Try first, then look.

***

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, LeaveOneOut, KFold, StratifiedKFold,
                                     cross_val_score, cross_validate, cross_val_predict)
from sklearn.metrics import mean_squared_error, confusion_matrix, RocCurveDisplay
from sklearn.metrics import roc_auc_score

## Setup

Run as given. It loads `Carseats`, creates the binary target `High` exactly as in Lab 5 (`Sales > 8`), encodes the categorical predictors as in Lab 7, Part 4, and makes one stratified 70/30 split. Use `X_tr`, `y_tr` for every choice, and touch `X_te`, `y_te` once, in 2(c).

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Carseats.csv'
carseats = pd.read_csv(url)
carseats.index = range(1, len(carseats) + 1)
carseats['High'] = (carseats['Sales'] > 8).astype(int)

X = pd.get_dummies(carseats.drop(columns=['Sales', 'High']), drop_first=True).astype(float)
y = carseats['High']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=2026)

print(X.columns.tolist())
print(f"Training rows: {len(y_tr)}, test rows: {len(y_te)}, share High: {y.mean():.2f}")

['CompPrice', 'Income', 'Advertising', 'Population', 'Price', 'Age', 'Education', 'ShelveLoc_Good', 'ShelveLoc_Medium', 'Urban_Yes', 'US_Yes']
Training rows: 280, test rows: 120, share High: 0.41


Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)` for every CV in this question.

***

**2(a)** Compare logistic regression (scaled, in a pipeline), LDA and Gaussian Naive Bayes by 5-fold CV AUC on the training set. Report each mean CV AUC and its standard error. Which differences are meaningful, which model do you carry forward, and why might Naive Bayes trail the other two on these predictors? **[3]** *(at most 3 sentences)*

In [10]:
#YOUR CODE HERE
# Use StratifiedKFold(n_splits=5, shuffle=True, random_state=2026) for every CV in this question.
skf = StratifiedKFold(n_splits=5,
                      shuffle=True,
                      random_state=2026)

# Pipelines/Estimators for each model
models = {
    'Logistic': Pipeline([
        ('scale', StandardScaler()),
        ('clf',   LogisticRegression())
    ]),
    'LDA': LinearDiscriminantAnalysis(),
    'Naive Bayes': GaussianNB()
}

# Loop through each model to calculate and print mean CV AUC and its standard error
for name, model in models.items():
    # cross_val_score returns an array of scores for each of the 5 folds
    scores = cross_val_score(model, X_tr, y_tr, cv=skf, scoring='roc_auc')
    mean_auc = scores.mean()
    # Standard error of the CV score: std with ddof=1 divided by the square root of the number of folds
    se_auc = scores.std(ddof=1) / np.sqrt(len(scores))
    print(f"{name:12} Mean CV AUC: {mean_auc:.4f} | SE: {se_auc:.4f}")

Logistic     Mean CV AUC: 0.9557 | SE: 0.0124
LDA          Mean CV AUC: 0.9520 | SE: 0.0139
Naive Bayes  Mean CV AUC: 0.8087 | SE: 0.0127


**2(a)**
Which differences are meaningful, which model do you carry forward, and why might Naive Bayes trail the other two on these predictors? [3] (at most 3 sentences)

The Logistic Regression, LDA, and Naive Bayes have the following average CV AUC, respectively, 0.9557, 0.9520, 0.8087, which I find the gap between Naive Bayes and the two other models the largest, whereas the gap of average CV AUC between Logistics regression and LDA to be neglibile; hence, I would like to carry Logistics Regression forward as it's less complex than LDA.

Naive Bayes perform the worst out of the three model as it assumes that all predictors are conditionally independent, whne in reality, comp price and price highly correlate with each other.


> 📍 **Where to look**
>
> - **Imports:** Lab 7, **Part 0: Setup and Data** (first code cell), plus `GaussianNB` from the Lab 6 Setup cell.
> - **Code:** Lab 7, **Part 4.1** (the `pipe_lr` pipeline and the `StratifiedKFold` splitter) and **4.2** (the demo with the `models` dictionary). Keep the fold scores instead of only their mean.
> - **SE:** Lab 8, **Part 4: Cross-Validation for Model Selection**, the cell starting `kf = KFold(` (the line `cv_se = ... std(..., ddof=1) / np.sqrt(...)`). Lab 9, **3.1 The CV Error Curve**, explains why `ddof=1`.
> - **Reasoning:** Lab 7, **Question 4.2**. For Naive Bayes: Lab 6, **Part 3: Within-Class Correlation Structure** and **Question 3.1**. Which `Carseats` prices move together?

***

**2(b)** Missing a high-sales store (a false negative) costs **3 times** as much as stocking a store that turns out to be low-sales (a false positive). State the cost-optimal threshold. Using out-of-fold predicted probabilities of your chosen model on the training set, report sensitivity and specificity at 0.5 and at your threshold, and explain the trade-off in one sentence. **[3]** *(at most 3 sentences)*

In [11]:
### YOUR CODE HERE
lr_model = Pipeline([
        ('scale', StandardScaler()),
        ('clf',   LogisticRegression())])

oof = {}
oof['Logistic'] = cross_val_predict(lr_model,
                                         X_tr,
                                         y_tr,
                                         cv=skf,
                                         method='predict_proba')[:, 1]

cost_optimal_threshold = 0.25
for threshold in (0.5, cost_optimal_threshold):
  pred = (oof['Logistic'] >= threshold).astype(int)
  cm = confusion_matrix(y_tr, pred); tn, fp, fn, tp = cm.ravel()

  sensitivity = tp/(fn+tp)
  specificity = tn/(fp+tn)
  accuracy = (tp+tn)/len(y_tr)

  table = pd.DataFrame({
      'Sensitivity' : sensitivity,
      'Specificity' : specificity,
      'Accuracy' : accuracy,
  },index=[0])
  print()
  print(f"The confusion matrix of the Logistic regression at threshold {threshold} is \n {cm}")
  print(table.to_string(index=False))



The confusion matrix of the Logistic regression at threshold 0.5 is 
 [[145  20]
 [ 17  98]]
 Sensitivity  Specificity  Accuracy
    0.852174     0.878788  0.867857

The confusion matrix of the Logistic regression at threshold 0.25 is 
 [[131  34]
 [  6 109]]
 Sensitivity  Specificity  Accuracy
    0.947826     0.793939  0.857143


**2(b)**

*Your answer here:*

The sensitivity, and specificity (sensitivity, specificity) of the Logistic Regression (the model I carried on) at threshold 0.5 and 0.25 respectively are (0.852174, 0.878788) and (0.947826, 0.793939); evidently, as we decreased the threshold from 0.5 to 0.25, sensitivity increased by 0.095652 and specificity decreased by 0.084849. As the cost of missing a high-sale store (false negative) is 3 times the cost of misidentifying a low-sale store to be high sale (false positive), it's desirable to improve in identifying an actual high-sale store to be high scale by increasing sensitivty. To do so, the threshold is lowered to increase sensitivity (true positive rate), but sacrificing some specificity (true negative rate).

> 📍 **Where to look**
>
> - **Threshold:** Lecture 5, **7.2 Threshold Selection**, and Lab 5, **4.2 Threshold selection** with **Question 4.2** (the lab used a cost ratio of 2; here it is 3).
> - **Code:** Lab 7, **Part 4.2**, the second demo cell (`cross_val_predict(..., method='predict_proba')` into `oof`), then **Exercise 4.2** (sensitivity and specificity at a chosen threshold).

***

**2(c)** Touch the test set once with your chosen model. Report the test AUC, and the sensitivity and specificity at your threshold from 2(b). Compare them with your training-CV figures. **[2]** *(at most 2 sentences)*

In [12]:
### YOUR CODE HERE
from sklearn.metrics import roc_auc_score

lr_model.fit(X_tr, y_tr)
test_probs = lr_model.predict_proba(X_te)[:, 1]

test_auc = roc_auc_score(y_te, test_probs)
print(f"Test ROC AUC: {test_auc:.4f}")

cost_optimal_threshold = 0.25
pred = (test_probs >= cost_optimal_threshold).astype(int)
cm = confusion_matrix(y_te, pred); tn, fp, fn, tp = cm.ravel()

sensitivity = tp/(fn+tp)
specificity = tn/(fp+tn)
accuracy = (tp+tn)/len(y_te)

table = pd.DataFrame({
      'Sensitivity' : sensitivity,
      'Specificity' : specificity,
      'Accuracy' : accuracy,
  },index=[0])

print(f"\nThe confusion matrix of the Logistic regression on test set at threshold {threshold} is \n {cm}")
print(table.to_string(index=False))

Test ROC AUC: 0.9638

The confusion matrix of the Logistic regression on test set at threshold 0.25 is 
 [[54 17]
 [ 2 47]]
 Sensitivity  Specificity  Accuracy
    0.959184     0.760563  0.841667


**2(c)**

*Your answer here:*

> 📍 **Where to look**
>
> - **Code:** Lab 9, **Exercise 4.2: One test touch, two final refits** (fit on the training set, score the test set once). `roc_auc_score` is in Lab 5, **Exercise 5.1**.
> - **Reasoning:** what "touch once" protects is in the Lab 9 introduction: split once, decide by CV on the training set, touch the test set once, refit on all the data.

***

**2(d)** Refit the full logistic regression from Lab 5 with `statsmodels` on all 400 stores. Report the odds ratio for a **$10** increase in `Price`, with its 95% confidence interval, and interpret it in one sentence. **[2]** *(at most 2 sentences)*

In [13]:
### YOUR CODE HERE
model_full = smf.logit("High ~ CompPrice + Income + Advertising + Population + Price + Age + Education + C(ShelveLoc) + C(Urban) + C(US)", data=carseats).fit()
print(model_full.summary())

log_odds_price = model_full.params['Price'].round(4)
odds_ratio_price = np.exp(log_odds_price).round(4)
odds_ratio_price_10 = np.exp(log_odds_price*10).round(4)
print(f"Log odds of price is {log_odds_price}, Odds ratio of price is {odds_ratio_price}, Odds ratio of $10 increase in price is {odds_ratio_price_10}")

conf_full = model_full.conf_int(alpha=0.05) # extract CI
conf_full.columns = ['CI lower', 'CI upper']

odds_ratio_table = pd.DataFrame({
    "Log Odds": model_full.params.round(4),
    "Odds_ratio": np.exp(model_full.params).round(4),
    "OR Lower CI": np.exp(conf_full['CI lower']),
    "OR Upper CI": np.exp(conf_full['CI upper'])
})
display(odds_ratio_table)

Optimization terminated successfully.
         Current function value: 0.226845
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                   High   No. Observations:                  400
Model:                          Logit   Df Residuals:                      388
Method:                           MLE   Df Model:                           11
Date:                Fri, 20 Nov 2026   Pseudo R-squ.:                  0.6649
Time:                        04:20:42   Log-Likelihood:                -90.738
converged:                       True   LL-Null:                       -270.74
Covariance Type:            nonrobust   LLR p-value:                 1.843e-70
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -5.0890      2.407     -2.114      0.034      -9.806      

,Log Odds,Odds_ratio,OR Lower CI,OR Upper CI
Intercept,-5.0890,0.0062,0.000055,0.689500
C(ShelveLoc)[T.Good],8.2409,3792.9056,500.984141,28715.744397
C(ShelveLoc)[T.Medium],3.5970,36.4903,10.166232,130.976870
C(Urban)[T.Yes],-0.3828,0.6820,0.288302,1.613129
C(US)[T.Yes],-0.6986,0.4973,0.154931,1.596087
CompPrice,0.1725,1.1882,1.135097,1.243875
Income,0.0341,1.0347,1.018816,1.050860
Advertising,0.2957,1.3441,1.215675,1.486165
Population,-0.0009,0.9991,0.996308,1.001866
Price,-0.1675,0.8458,0.813865,0.878951


**2(d)**

*Your answer here:*

The coefficient of predictor Price is the change in log-odds per unit of change in X; each additional unit of X multiplies the odd of an event occuring (in our case, High Sales) by odds ratio which is the exponential of logodds. Log odds of price is -0.1675, Odds ratio of price is 0.8458, Odds ratio of $10 increase in price is 0.1873; hence, a 10 dollar odds increase in Price results the probability of a specific store being considered as "High Sale" is multiplied by 0.1873

> 📍 **Where to look**
>
> - **Formula and fit:** Lab 5, **Exercise 1.2a** (the full-model formula with `C(ShelveLoc)`, `C(Urban)`, `C(US)`).
> - **Odds ratio and CI:** Lab 5, **1.1 Simple logistic regression**, the cell `# Odds ratio and 95% CI for Price`, and **Exercise 1.2b**. For a $10 change, multiply the coefficient and both CI limits by 10 before exponentiating.
> - **Wording:** Lab 5, **Question 1.1**: odds, not probability.

***

## After you finish

Check yourself against the lab:

1. Did every figure you quoted come from your own printed output?
2. Did every part get a written answer, not only code?
3. Which parts needed the lab open? Those cells are the ones to rehearse before the exam.